# Understanding Supervised Fine-Tuning (SFT)

This notebook demonstrates how supervised fine-tuning works for language models.

## What is SFT?

Supervised Fine-Tuning (SFT) is the process of adapting a pre-trained language model to a specific task by training it on instruction-following examples. The model learns to:
- Follow instructions
- Generate appropriate responses
- Adapt to specific domains or styles

## Key Concepts

1. **Next-token prediction**: Model predicts the next token given previous context
2. **Prompt masking**: We only compute loss on response tokens, not the prompt
3. **LoRA**: Efficient fine-tuning by training low-rank adapters

Let's see it in action!

In [ ]:
# Setup
import sys
sys.path.append('..')

import torch
from transformers import TrainingArguments
from datasets import Dataset

from src.models.language import LanguageModel
from src.data.processors.text import TextProcessor
from src.core.sft.trainer import SFTTrainer
from src.core.sft.collator import DataCollatorForSFT, create_sft_dataset
from src.evaluation.metrics.text import TextMetrics

## Step 1: Load a Small Model

We'll use GPT-2 (124M parameters) with LoRA for efficient fine-tuning.

In [ ]:
# Load model with LoRA
model = LanguageModel.from_pretrained(
    "gpt2",
    use_lora=True,
    lora_config={
        "r": 8,
        "lora_alpha": 16,
        "lora_dropout": 0.05,
    }
)

print(f"Total parameters: {model.num_parameters:,}")
print(f"Trainable parameters: {model.num_trainable_parameters:,}")
print(f"Trainable %: {100 * model.num_trainable_parameters / model.num_parameters:.2f}%")

## Step 2: Create a Training Dataset

We'll create a small dataset of instruction-response pairs.

In [ ]:
# Create training data
train_examples = [
    {"prompt": "What is the capital of France?", "response": "The capital of France is Paris."},
    {"prompt": "What is 2 + 2?", "response": "2 + 2 equals 4."},
    {"prompt": "Who wrote Romeo and Juliet?", "response": "William Shakespeare wrote Romeo and Juliet."},
    {"prompt": "What is the largest planet?", "response": "Jupiter is the largest planet in our solar system."},
    {"prompt": "What is Python?", "response": "Python is a high-level programming language known for its simplicity."},
]

eval_examples = [
    {"prompt": "What is the capital of Spain?", "response": "The capital of Spain is Madrid."},
    {"prompt": "What is 5 + 3?", "response": "5 + 3 equals 8."},
]

print(f"Training examples: {len(train_examples)}")
print(f"Eval examples: {len(eval_examples)}")

## Step 3: Tokenize the Data

Convert text to token IDs and create labels with prompt masking.

In [ ]:
# Tokenize datasets
train_tokenized = create_sft_dataset(
    examples=train_examples,
    tokenizer=model.tokenizer,
    max_length=128,
)

eval_tokenized = create_sft_dataset(
    examples=eval_examples,
    tokenizer=model.tokenizer,
    max_length=128,
)

# Convert to Dataset
train_dataset = Dataset.from_list(train_tokenized)
eval_dataset = Dataset.from_list(eval_tokenized)

print("Example tokenized data:")
example = train_tokenized[0]
print(f"Input IDs length: {len(example['input_ids'])}")
print(f"Labels (first 20): {example['labels'][:20]}")
print("Note: -100 means 'ignore in loss computation' (prompt tokens)")

## Step 4: Setup Training

Create the data collator and training arguments.

In [ ]:
# Data collator
data_collator = DataCollatorForSFT(
    tokenizer=model.tokenizer,
    max_length=128,
)

# Training arguments
training_args = TrainingArguments(
    output_dir="./outputs/sft_tutorial",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=5e-5,
    warmup_steps=10,
    logging_steps=1,
    eval_steps=5,
    evaluation_strategy="steps",
    save_strategy="no",  # Don't save checkpoints for this demo
)

print("Training configuration:")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Learning rate: {training_args.learning_rate}")

## Step 5: Create Trainer and Train

Use our custom SFT trainer with detailed logging.

In [ ]:
# Create trainer
trainer = SFTTrainer(
    model=model.model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=model.tokenizer,
    data_collator=data_collator,
    loss_type="causal_lm",
    log_predictions=False,  # Disable for this demo
)

print("Starting training...")
trainer.train()
print("Training complete!")

## Step 6: Evaluate the Model

Let's see how the model performs after training.

In [ ]:
# Evaluate
metrics = trainer.evaluate()

print("\nEvaluation Metrics:")
for key, value in metrics.items():
    print(f"{key}: {value:.4f}")

## Step 7: Generate Responses

Test the fine-tuned model with some prompts.

In [ ]:
# Test generation
model.eval()

test_prompts = [
    "What is the capital of Italy?",
    "What is 10 + 5?",
    "Who invented the telephone?",
]

processor = TextProcessor(model.tokenizer, max_length=128)

print("\nGenerated Responses:")
print("=" * 60)

for prompt in test_prompts:
    # Encode
    encoded = processor.tokenize(prompt, return_tensors="pt")
    encoded = {k: v.to(model.device) for k, v in encoded.items()}
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            encoded["input_ids"],
            attention_mask=encoded["attention_mask"],
            max_new_tokens=30,
            temperature=0.7,
            do_sample=True,
        )
    
    # Decode
    generated_text = processor.decode(outputs[0], skip_special_tokens=True)
    
    print(f"\nPrompt: {prompt}")
    print(f"Generated: {generated_text}")
    print("-" * 60)

## Step 8: Analyze Training Metrics

Visualize how the model improved during training.

In [ ]:
import matplotlib.pyplot as plt

# Get training metrics
metrics = trainer.get_training_metrics()

if metrics['steps']:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    # Loss
    if metrics['losses']:
        axes[0, 0].plot(metrics['steps'], metrics['losses'])
        axes[0, 0].set_title('Training Loss')
        axes[0, 0].set_xlabel('Step')
        axes[0, 0].set_ylabel('Loss')
    
    # Accuracy
    if metrics['accuracies']:
        axes[0, 1].plot(metrics['steps'], metrics['accuracies'])
        axes[0, 1].set_title('Token Accuracy')
        axes[0, 1].set_xlabel('Step')
        axes[0, 1].set_ylabel('Accuracy')
    
    # Perplexity
    if metrics['perplexities']:
        axes[1, 0].plot(metrics['steps'], metrics['perplexities'])
        axes[1, 0].set_title('Perplexity')
        axes[1, 0].set_xlabel('Step')
        axes[1, 0].set_ylabel('Perplexity')
    
    # Gradient Norm
    if metrics['grad_norms']:
        axes[1, 1].plot(metrics['steps'], metrics['grad_norms'])
        axes[1, 1].set_title('Gradient Norm')
        axes[1, 1].set_xlabel('Step')
        axes[1, 1].set_ylabel('Grad Norm')
    
    plt.tight_layout()
    plt.show()
else:
    print("No metrics to plot")

## Summary

In this notebook, we:
1. ✅ Loaded GPT-2 with LoRA adapters (only 0.5% of parameters trainable)
2. ✅ Created a small instruction-following dataset
3. ✅ Tokenized data with proper prompt masking
4. ✅ Trained the model using SFT
5. ✅ Evaluated performance
6. ✅ Generated responses from the fine-tuned model
7. ✅ Analyzed training dynamics

## Key Takeaways

- **SFT is straightforward**: Just next-token prediction on instruction data
- **Prompt masking is crucial**: Only compute loss on responses
- **LoRA is efficient**: Train only 0.5% of parameters
- **Small datasets work**: Even 5 examples show improvement

## Next Steps

- Try with larger datasets (Alpaca, Dolly, etc.)
- Experiment with different LoRA ranks
- Try different prompt templates
- Move to reward modeling and RLHF!

See `02_reward_modeling.ipynb` for the next technique.

---

# Advanced Examples: Next Steps

Now let's explore the three advanced topics mentioned above with working code examples!

---

# Advanced Examples: Next Steps

Now let's explore the three advanced topics mentioned above with working code examples!

## Example 1: Using Larger Datasets

Let's load a real instruction-following dataset from HuggingFace Hub.

We'll use a subset of popular datasets:
- **Alpaca**: Stanford's instruction-following dataset (52K examples)
- **Dolly**: Databricks' high-quality instruction dataset (15K examples)
- **OpenAssistant**: Conversations from open-assistant.io (100K+ examples)

For this demo, we'll load a small sample to keep it fast.

In [ ]:
from src.data.loaders import load_dataset as load_hf_dataset
from src.data.processors.text import create_prompt_template

# Option 1: Load Alpaca-style dataset
# Uncomment to try (requires internet)
# alpaca = load_hf_dataset("tatsu-lab/alpaca")
# print(f"Alpaca dataset size: {len(alpaca['train'])}")

# Option 2: Load Dolly dataset
# dolly = load_hf_dataset("databricks/databricks-dolly-15k")
# print(f"Dolly dataset size: {len(dolly['train'])}")

# For this demo, let's create a larger synthetic dataset
# In practice, you'd use one of the real datasets above

# Simulated larger dataset
categories = {
    "geography": [
        ("What is the capital of {country}?", "The capital of {country} is {capital}."),
        ("Where is {country} located?", "{country} is located in {region}."),
    ],
    "math": [
        ("What is {a} + {b}?", "{a} + {b} equals {result}."),
        ("Calculate {a} * {b}", "{a} * {b} equals {result}."),
    ],
    "science": [
        ("What is {concept}?", "{concept} is {definition}."),
        ("Explain {topic}", "{topic}: {explanation}"),
    ],
}

# Generate examples
larger_dataset = []

# Geography examples
countries = [("France", "Paris", "Europe"), ("Japan", "Tokyo", "Asia"), 
             ("Brazil", "Brasília", "South America"), ("Egypt", "Cairo", "Africa")]
for country, capital, region in countries:
    larger_dataset.append({
        "prompt": f"What is the capital of {country}?",
        "response": f"The capital of {country} is {capital}."
    })
    larger_dataset.append({
        "prompt": f"Where is {country} located?",
        "response": f"{country} is located in {region}."
    })

# Math examples  
import random
for _ in range(10):
    a, b = random.randint(1, 20), random.randint(1, 20)
    larger_dataset.append({
        "prompt": f"What is {a} + {b}?",
        "response": f"{a} + {b} equals {a+b}."
    })
    larger_dataset.append({
        "prompt": f"Calculate {a} * {b}",
        "response": f"{a} * {b} equals {a*b}."
    })

# Science examples
science_facts = [
    ("photosynthesis", "the process by which plants convert light into energy"),
    ("DNA", "the molecule that carries genetic information"),
    ("gravity", "the force that attracts objects toward each other"),
    ("evolution", "the process by which species change over time"),
]
for concept, definition in science_facts:
    larger_dataset.append({
        "prompt": f"What is {concept}?",
        "response": f"{concept.capitalize()} is {definition}."
    })

print(f"Generated dataset size: {len(larger_dataset)} examples")
print(f"\nSample examples:")
for i in range(3):
    print(f"\nExample {i+1}:")
    print(f"  Prompt: {larger_dataset[i]['prompt']}")
    print(f"  Response: {larger_dataset[i]['response']}")

### Training on Larger Dataset

Now let's train on our larger dataset and see how performance improves.

In [ ]:
# Split into train/eval
train_size = int(0.9 * len(larger_dataset))
large_train = larger_dataset[:train_size]
large_eval = larger_dataset[train_size:]

print(f"Training examples: {len(large_train)}")
print(f"Eval examples: {len(large_eval)}")

# Tokenize
large_train_tokenized = create_sft_dataset(
    examples=large_train,
    tokenizer=model.tokenizer,
    max_length=128,
)

large_eval_tokenized = create_sft_dataset(
    examples=large_eval,
    tokenizer=model.tokenizer,
    max_length=128,
)

large_train_dataset = Dataset.from_list(large_train_tokenized)
large_eval_dataset = Dataset.from_list(large_eval_tokenized)

print("\n✅ Dataset prepared for training!")
print("\nTip: With more data, you can:")
print("  - Train for fewer epochs (1-2 instead of 3)")
print("  - Use larger batch sizes for faster training")
print("  - Expect better generalization to unseen prompts")

## Example 2: Experimenting with Different LoRA Ranks

LoRA rank (`r`) controls the capacity of the adapters:
- **Lower rank (r=4, 8)**: Fewer parameters, faster training, less capacity
- **Higher rank (r=16, 32, 64)**: More parameters, slower training, more capacity

Let's compare different ranks!

In [ ]:
# Compare different LoRA ranks
lora_configs = {
    "Very Low (r=4)": {"r": 4, "lora_alpha": 8, "lora_dropout": 0.05},
    "Low (r=8)": {"r": 8, "lora_alpha": 16, "lora_dropout": 0.05},
    "Medium (r=16)": {"r": 16, "lora_alpha": 32, "lora_dropout": 0.1},
    "High (r=32)": {"r": 32, "lora_alpha": 64, "lora_dropout": 0.1},
    "Very High (r=64)": {"r": 64, "lora_alpha": 128, "lora_dropout": 0.1},
}

print("LoRA Rank Comparison for GPT-2 (124M parameters):\n")
print(f"{'Config':<20} {'Trainable Params':<20} {'% of Total':<15} {'Memory Impact'}")
print("=" * 75)

for name, config in lora_configs.items():
    # Load model with this config
    test_model = LanguageModel.from_pretrained(
        "gpt2",
        use_lora=True,
        lora_config=config,
    )
    
    trainable = test_model.num_trainable_parameters
    percent = 100 * trainable / test_model.num_parameters
    
    # Memory impact (rough estimate)
    if config['r'] <= 8:
        memory = "Low"
    elif config['r'] <= 16:
        memory = "Medium"
    elif config['r'] <= 32:
        memory = "High"
    else:
        memory = "Very High"
    
    print(f"{name:<20} {trainable:>18,} {percent:>13.2f}% {memory:>15}")
    
    # Clean up
    del test_model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("\n💡 Recommendations:")
print("  • Start with r=8 or r=16 (good balance)")
print("  • Use r=4 for very fast experimentation")
print("  • Use r=32+ for complex tasks or large models")
print("  • Higher rank = more capacity but diminishing returns")

### Training with Different Ranks

Let's train with a different rank and compare results.

In [ ]:
# Load model with higher rank
model_high_rank = LanguageModel.from_pretrained(
    "gpt2",
    use_lora=True,
    lora_config={
        "r": 32,  # Higher rank for more capacity
        "lora_alpha": 64,
        "lora_dropout": 0.1,
        "target_modules": ["c_attn", "c_proj"],  # More modules
    }
)

print(f"High-rank model:")
print(f"  Trainable parameters: {model_high_rank.num_trainable_parameters:,}")
print(f"  Trainable %: {model_high_rank.percent_trainable:.2f}%")
print(f"\nCompare with original (r=8):")
print(f"  Trainable parameters: {model.num_trainable_parameters:,}")
print(f"  Trainable %: {model.percent_trainable:.2f}%")
print(f"\nIncrease: {model_high_rank.num_trainable_parameters / model.num_trainable_parameters:.1f}x more parameters")

print("\n🎯 Expected Outcomes:")
print("  • r=32 should learn faster and achieve lower loss")
print("  • r=32 may generalize better on complex tasks")
print("  • r=32 requires ~2x more memory and compute")
print("  • For simple tasks, r=8 might be sufficient!")

## Example 3: Different Prompt Templates

The prompt template affects how instructions are formatted. Different templates work better for different models:

- **Alpaca**: Standard instruction format with clear sections
- **ChatML**: Conversational format with special tokens
- **LLaMA-2**: LLaMA-2 specific format with system prompts
- **Plain**: Simple concatenation (basic)

Let's see how they differ!

In [ ]:
from src.data.processors.text import create_prompt_template

# Sample instruction
instruction = "Explain what machine learning is."
input_text = "Keep it simple for a beginner."
response = "Machine learning is a branch of AI where computers learn from data without being explicitly programmed."

# Try different templates
templates = ["alpaca", "chatml", "llama2", "plain"]

print("Comparing Prompt Templates:\n")
print("="*80)

for template_name in templates:
    template = create_prompt_template(template_name)
    formatted = template(instruction, input_text)
    
    # Add response for full example
    full_text = formatted + response
    
    print(f"\n### {template_name.upper()} Template ###")
    print(f"Format: {full_text}")
    print(f"\nLength: {len(full_text)} chars, ~{len(full_text.split())} tokens")
    print("-"*80)

### Training with Different Templates

Let's prepare datasets with different templates and see which works best.

In [ ]:
# Create datasets with different templates
def prepare_dataset_with_template(examples, template_name):
    """Prepare dataset with specific prompt template."""
    template = create_prompt_template(template_name)
    
    formatted_examples = []
    for ex in examples:
        # Format prompt with template
        formatted_prompt = template(ex['prompt'], "")
        formatted_examples.append({
            "prompt": formatted_prompt,
            "response": ex['response']
        })
    
    return formatted_examples

# Use our original small dataset
original_examples = [
    {"prompt": "What is the capital of France?", "response": "The capital of France is Paris."},
    {"prompt": "What is 2 + 2?", "response": "2 + 2 equals 4."},
    {"prompt": "Who wrote Romeo and Juliet?", "response": "William Shakespeare wrote Romeo and Juliet."},
]

# Compare templates
print("Template Comparison:\n")

for template_name in ["alpaca", "plain"]:
    formatted = prepare_dataset_with_template(original_examples, template_name)
    
    print(f"### {template_name.upper()} Template ###")
    print(f"Example 1 (formatted prompt):")
    print(f"{formatted[0]['prompt']}")
    print(f"Response: {formatted[0]['response']}")
    print()

print("\n💡 Template Selection Tips:")
print("  • Alpaca: Best for instruction-following tasks")
print("  • ChatML: Good for conversational models")
print("  • LLaMA-2: Required for LLaMA-2 models")
print("  • Plain: Simple baseline, works with any model")
print("\n  ⚠️  Consistency is key: Use the same template for training and inference!")

## Bonus: Combining Everything

Now let's combine all three improvements:
1. ✅ Larger dataset
2. ✅ Optimal LoRA rank
3. ✅ Best prompt template

This is what a production training run might look like!

In [ ]:
print("🚀 Production-Ready Configuration:\n")

# 1. Load model with optimal LoRA config
production_model = LanguageModel.from_pretrained(
    "gpt2",
    use_lora=True,
    lora_config={
        "r": 16,  # Good balance
        "lora_alpha": 32,
        "lora_dropout": 0.1,
        "target_modules": ["c_attn", "c_proj"],
    }
)

print(f"✅ Model: GPT-2 with LoRA (r=16)")
print(f"   Trainable: {production_model.num_trainable_parameters:,} params")

# 2. Prepare larger dataset with Alpaca template
production_data = prepare_dataset_with_template(larger_dataset, "alpaca")
print(f"\n✅ Dataset: {len(production_data)} examples with Alpaca template")

# 3. Split and tokenize
train_size = int(0.9 * len(production_data))
prod_train = production_data[:train_size]
prod_eval = production_data[train_size:]

prod_train_tokenized = create_sft_dataset(
    examples=prod_train,
    tokenizer=production_model.tokenizer,
    max_length=256,  # Longer for Alpaca template
)

prod_eval_tokenized = create_sft_dataset(
    examples=prod_eval,
    tokenizer=production_model.tokenizer,
    max_length=256,
)

prod_train_dataset = Dataset.from_list(prod_train_tokenized)
prod_eval_dataset = Dataset.from_list(prod_eval_tokenized)

print(f"   Train: {len(prod_train_dataset)} examples")
print(f"   Eval: {len(prod_eval_dataset)} examples")

# 4. Optimized training arguments
prod_training_args = TrainingArguments(
    output_dir="./outputs/production_sft",
    num_train_epochs=2,  # Fewer epochs with more data
    per_device_train_batch_size=4,  # Larger batch
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,  # Effective batch = 8
    learning_rate=3e-4,  # Higher LR with more data
    warmup_steps=50,
    logging_steps=10,
    eval_steps=50,
    evaluation_strategy="steps",
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

print(f"\n✅ Training Config:")
print(f"   Epochs: {prod_training_args.num_train_epochs}")
print(f"   Batch size: {prod_training_args.per_device_train_batch_size} (effective: {prod_training_args.per_device_train_batch_size * prod_training_args.gradient_accumulation_steps})")
print(f"   Learning rate: {prod_training_args.learning_rate}")

print("\n✅ Ready for production training!")
print("\nTo train, run:")
print("""trainer = SFTTrainer(
    model=production_model.model,
    args=prod_training_args,
    train_dataset=prod_train_dataset,
    eval_dataset=prod_eval_dataset,
    tokenizer=production_model.tokenizer,
    data_collator=DataCollatorForSFT(production_model.tokenizer, max_length=256),
)
trainer.train()""")

## Summary: What We Learned

### 1. Larger Datasets
- ✅ More data = better generalization
- ✅ Can use fewer epochs (1-2 vs 3)
- ✅ Popular datasets: Alpaca (52K), Dolly (15K), OpenAssistant (100K+)
- ⚠️ More data needs more compute time

### 2. LoRA Rank Selection
- ✅ r=4-8: Fast experimentation, simple tasks
- ✅ r=16: Sweet spot for most tasks
- ✅ r=32-64: Complex tasks, large models
- ⚠️ Higher rank = more memory and compute

### 3. Prompt Templates
- ✅ Alpaca: Best for instructions
- ✅ ChatML: Conversational AI
- ✅ LLaMA-2: Model-specific
- ✅ Plain: Universal baseline
- ⚠️ Always use the same template for training and inference!

### Production Recipe
```python
# Best practices for production SFT:
1. Dataset: 10K-100K high-quality examples
2. LoRA: r=16, alpha=32, dropout=0.1
3. Template: Alpaca (or model-specific)
4. Batch: 4-8 per device (adjust for GPU memory)
5. Epochs: 1-3 (fewer with more data)
6. LR: 3e-4 to 5e-5 (higher for more data)
```

---

## Next: Phase 3 - Reward Modeling

Now that you understand SFT thoroughly, you're ready for Phase 3!

In the next phase, we'll learn:
- How to train reward models from preference data
- Bradley-Terry ranking loss
- Preparing for RLHF (PPO/DPO)

See you in Phase 3! 🚀